# 26 — Replikasi seed R512: G3TI_R512 dan G3TI_noT_R512, seed 12 dan 13, p100

Tujuan (pengguna: "lanjutkan", 3 Okt 2026). Kernel 25 (seed 11) memberi KID buah di bawah V0 untuk kedua arm R512.
Pada tampilan 512, arm tanpa diskriminator tekstur punya artefak kroma dua kali lipat (Nyquist R−Y 0,121 vs 0,066).
Kedua klaim itu butuh replikasi seed sebelum dipakai.

Pool 512, konfigurasi dan ekspor identik dengan kernel 25; hanya seed yang berubah
(`seed`, `controlled_init_seed`, `texture_seed` = seed×1000+1). GPU0 menjalankan G3TI_R512 s12 lalu s13, GPU1 menjalankan
G3TI_noT_R512 s12 lalu s13. Tidak ada detektor. Test tidak dibaca.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/g3ti_r512_seeds_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
# Study (p100 crops and configs) from kernel 13 output; three G3 tuning configs written next to G0-G3.
if torch.cuda.device_count()<1: raise RuntimeError('GPU required')
print('GPUs', torch.cuda.device_count())
source=unique('study.json')
STUDY=WORK/'study'
shutil.copytree(source,STUDY)
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)
P100=STUDY/'p100'
sys.path.insert(0,str(HERE))
from melon_cyclegan import texture_study as s
from melon_cyclegan import texture_study_eval as e
scope,m=s.verify_scope(STUDY,100,DATASET)
g0=json.loads((P100/'configs/G0.json').read_text()); g3=json.loads((P100/'configs/G3.json').read_text())
assert g0['subset_digest']==g3['subset_digest']==m['subset_digest']
# R512 pool: same crop windows, same reviewed-pool membership and same GrabCut masks (nearest-upsampled),
# images re-cropped from the original photographs at 512 px. Only resolution changes.
def build_pool512(src, dst, dataset, size=512, require_train=True):
    import csv, json, collections, hashlib
    from pathlib import Path
    import numpy as np, cv2
    from PIL import Image
    from melon_cyclegan import config as cfg
    from melon_cyclegan.crops import crop_with_reflection
    from melon_cyclegan.color import phi_numpy
    from melon_cyclegan.training_data import digest, verify, sha
    src, dst, dataset = Path(src), Path(dst), Path(dataset)
    if dst.exists(): raise ValueError(f'{dst} exists')
    man = json.loads((src/'crops_manifest.json').read_text())
    rows = {r['id']: r for r in csv.DictReader(open(dataset/'manifest.csv'))}
    k = size/float(man.get('size', 256)); records=[]; feats=collections.defaultdict(list); drift=[]
    for r in man['records']:
        row = rows[r['id']]
        if (require_train and row['split']!='train') or sha(dataset/row['image'])!=r['source_sha256']: raise ValueError(f"{r['id']}: source photo differs")
        with Image.open(dataset/row['image']) as h: img=h.convert('RGB')
        left, top, side = r['crop_window']
        crop = cv2.resize(crop_with_reflection(img,left,top,side),(size,size),interpolation=cv2.INTER_AREA)
        m256 = Image.open(src/r['mask']).convert('L')
        mask = np.asarray(m256.resize((size,size),Image.Resampling.NEAREST))
        old = np.asarray(Image.open(src/r['image']).convert('RGB'),np.float64)
        drift.append(np.abs(cv2.resize(crop,old.shape[1::-1],interpolation=cv2.INTER_AREA).astype(np.float64)-old).mean())
        rec = dict(r, crop_box=[float(v)*k for v in r['crop_box']])
        for key, arr in (('image', crop), ('mask', mask)):
            p = dst/r[key]; p.parent.mkdir(parents=True, exist_ok=True); Image.fromarray(arr).save(p); rec[f'{key}_sha256']=sha(p)
        feats[int(r['class_id'])].append(phi_numpy(crop.astype(np.float64)/255.0, mask>127)); records.append(rec)
    if max(drift) > 4.0: raise ValueError(f'512 crops do not match the 256 pool geometry (max mean abs diff {max(drift):.2f})')
    classes = {}
    for cid, v in sorted(feats.items()):
        x=np.stack(v); classes[cfg.CLASS_NAMES[cid]]=dict(class_id=cid,count=len(x),centroid=x.mean(0).tolist(),spread=x.std(0).tolist())
    new = {k2: v for k2, v in man.items() if k2 not in ('records','crops_digest','centroids_sha256')}
    new.update(records=records, size=size, crops_digest=digest(records), source_crops_digest_256=man['crops_digest'],
               note='R512: same windows, members and GrabCut masks as the 256 pool; images re-cropped at 512')
    (dst/'color_centroids.json').write_text(json.dumps(dict(version=cfg.VERSION, phi=man['phi'],
        source='curated train split real crops (R512)', crops_digest=new['crops_digest'], classes=classes), indent=2)+'\n')
    new['centroids_sha256'] = sha(dst/'color_centroids.json')
    (dst/'crops_manifest.json').write_text(json.dumps(new, indent=2)+'\n')
    verify(dst, man['training_policy'])
    return new, dict(max_drift=float(max(drift)), mean_drift=float(np.mean(drift)), n=len(records))

POOL512=P100/'crops512'
pool_manifest,pool_check=build_pool512(P100/'crops',POOL512,DATASET)
print('R512 pool',pool_check,pool_manifest['crops_digest'])
sys.path.insert(0,str(HERE/'tools'))
from calibrate_color_margin import calibrate
report=calibrate(POOL512); (POOL512/'color_margin.json').write_text(json.dumps(report,indent=2)+'\n')
print('margin 512',report['margin'],'margin 256',json.loads((P100/'crops/color_margin.json').read_text())['margin'])
if report['margin']<=0: raise ValueError('nonpositive margin at 512')
from melon_cyclegan.dataset_impl import load_pair
from melon_cyclegan.texture import patch_indices
for r in pool_manifest['records']:
    _,mk=load_pair(POOL512/r['image'],POOL512/r['mask']); patch_indices(mk[None],64,32,4,.95,random=False)
g3ti=dict(g3,composite_mask=0,lambda_identity=0.5,load_size=512,crop_size=512,texture_size=64,texture_stride=32,
          expected_crops_digest=pool_manifest['crops_digest'])
EXTRA={}
for seed in (12,13):
    for arm,upd in (('G3TI_R512',{}),('G3TI_noT_R512',{'lambda_texture':0.0})):
        EXTRA[f'{arm}_s{seed}']=dict(g3ti,**upd,seed=seed,controlled_init_seed=seed,texture_seed=seed*1000+1,
            experiment=f'p100_{arm}_seed{seed}',rung=f'{arm}_s{seed}',purpose=f'{arm} seed repeat; train only',
            note='identical to kernel 25 except seed')
for arm,cfg_ in EXTRA.items():
    target=P100/'configs'/f'{arm}.json'
    if target.exists(): raise ValueError(f'{target} exists')
    target.write_text(json.dumps(cfg_,indent=2)+'\n')
    print(arm,{k:cfg_[k] for k in ('model','seed','composite_mask','lambda_identity','lambda_texture','crop_size','texture_size','texture_stride')})


In [ ]:
# Train: GPU0 runs G3TI_R512 s12 then s13, GPU1 runs G3TI_noT_R512 s12 then s13 (sequential on one GPU if only one is present).
import shlex, time
def train_cmd(arm):
    return [sys.executable,str(HERE/'tools/train_cyclegan.py'),'--config',str(P100/'configs'/f'{arm}.json'),
            '--dataroot',str(POOL512),'--checkpoints-dir',str(P100/'checkpoints'),'--','--time_budget_minutes','120']
(P100/'logs').mkdir(exist_ok=True)
def chain(arms,gpu):
    env=(f"CUDA_VISIBLE_DEVICES={gpu} PYTHONPATH={shlex.quote(str(HERE))} MELON_DATASET={shlex.quote(str(DATASET))} "
         f"MELON_ASSETS={shlex.quote(str(ASSETS))} OMP_NUM_THREADS=2")
    cmd=' && '.join(env+' '+shlex.join(train_cmd(a))+' > '+shlex.quote(str(P100/'logs'/f'{a}.log'))+' 2>&1' for a in arms)
    return subprocess.Popen(cmd,shell=True,cwd=str(HERE))
started=time.time()
plan=[(['G3TI_R512_s12','G3TI_R512_s13'],0),(['G3TI_noT_R512_s12','G3TI_noT_R512_s13'],1)] if torch.cuda.device_count()>=2 else [(list(EXTRA),0)]
codes=[p.wait() for p in [chain(a,g) for a,g in plan]]
print('return codes',codes,'wall minutes',round((time.time()-started)/60,1))
for arm in EXTRA: print('---',arm); print((P100/'logs'/f'{arm}.log').read_text()[-1500:])
if any(codes): raise SystemExit('training failed')
for arm,c in EXTRA.items():
    loss=json.loads((P100/'checkpoints'/c['experiment']/'loss.json').read_text())
    if loss['stopped'] is not None or loss['epochs']!=c['n_epochs']+c['n_epochs_decay']: raise SystemExit(f'{arm} partial run')

# Pre-registered H-T check: mean losses over epochs 11-20 (LSGAN equilibrium 0.25).
import re
LOSS_MEANS={}
for arm in EXTRA:
    rows=[dict((k,float(v)) for k,v in re.findall(r'(\w+):([-\d.]+)',l))
          for l in (P100/'logs'/f'{arm}.log').read_text().splitlines()
          if (mm:=re.match(r'epoch (\d+) iter',l)) and int(mm.group(1))>=11]
    LOSS_MEANS[arm]={k:round(sum(r[k] for r in rows)/len(rows),3) for k in ('D_A','D_B','G_A','G_B','idt_A','idt_B','texture_D_A','texture_D_B') if k in rows[0]}
    LOSS_MEANS[arm]['n_rows']=len(rows)
    print(arm,LOSS_MEANS[arm])


In [ ]:
# Export: EMA G_B, translated at 512, pasted into the original photograph through the (upsampled) fruit mask.
# The exporter scales crop_box by 512/256 for the review bbox; at 512 crops the factor must be 1, fixed below.
gbc=s.tool('generate_balance_candidates')
for arm,c in EXTRA.items():
    out=P100/'candidates'/arm
    gbc.run(P100/'checkpoints',c['experiment'],'ema',POOL512,out,DATASET,'cuda')
    recs=json.loads((out/'records.json').read_text()); byid={r['id']:r for r in pool_manifest['records']}
    for r in recs:
        x,y,w,h=byid[r['parent_id']]['crop_box']; r['bbox']=[int(x),int(y),int(w),int(h)]; r['trace']['resolution']=512
    (out/'records.json').write_text(json.dumps(recs,indent=2)+'\n')
    s.write(out/'binding.json',dict(arm=arm,subset_digest=m['subset_digest'],pool512_digest=pool_manifest['crops_digest'],
        generator_sha256=s.sha(P100/'checkpoints'/c['experiment']/'latest_net_G_B_ema.pth'),
        config_sha256=s.sha(P100/'configs'/f'{arm}.json'),weights='ema'))
    print(arm,len(recs),'candidates')


In [ ]:
# Keep only what local FID/KID, review and the paper need; drop crops, full frames and other arms.
OUT=WORK/'result'; OUT.mkdir()
for arm,c in EXTRA.items():
    ck=P100/'checkpoints'/c['experiment']; dst=OUT/'checkpoints'/c['experiment']; dst.mkdir(parents=True)
    for name in ('loss.json','melon_identity.json','latest_net_G_B_ema.pth','latest_net_G_A_ema.pth'):
        if (ck/name).exists(): shutil.copy2(ck/name,dst/name)
    cand=P100/'candidates'/arm; cdst=OUT/'candidates'/arm; cdst.mkdir(parents=True)
    for sub in ('images','masks'): shutil.copytree(cand/sub,cdst/sub)
    for name in ('records.json','binding.json'): shutil.copy2(cand/name,cdst/name)
    shutil.copy2(P100/'configs'/f'{arm}.json',OUT/f'config_{arm}.json')
    shutil.copy2(P100/'logs'/f'{arm}.log',OUT/f'train_{arm}.log')
for name in ('crops_manifest.json','color_centroids.json','color_margin.json'): shutil.copy2(POOL512/name,OUT/f'pool512_{name}')
(OUT/'summary.json').write_text(json.dumps(dict(arms=list(EXTRA),pool512=pool_check,loss_means_ep11_20=LOSS_MEANS,subset_digest=m['subset_digest'],
    test_used=False,wall_minutes=round((time.time()-started)/60,1)),indent=1))
shutil.rmtree(STUDY,ignore_errors=True); shutil.rmtree(WORK/'code',ignore_errors=True)
total=sum(p.stat().st_size for p in WORK.rglob('*') if p.is_file())
print('output files',sum(1 for p in WORK.rglob('*') if p.is_file()),'MB',round(total/2**20))
